# Algorithmic Trading with Python & Google Colab
## Session 1: Can Markets Be Predicted? From Efficient Markets to Trading Signals

*(c) Dr. Yves J. Hilpisch | The Python Quants GmbH | https://tpq.io | https://hilpisch.com*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yveshilpisch/pyalgo/blob/main/notebooks/01_can_markets_be_predicted.ipynb)

---

### Objectives
1. **Google Colab Environment**: Research workflow, CPU runtime setup, and reproducibility.
2. **Efficient Market Hypothesis (EMH)**: Simulating random walks (Geometric Brownian Motion) vs. empirical market series.
3. **Statistical Tests of Predictability**: Autocorrelation, Ljung-Box test, and rolling autocorrelation regimes.
4. **Signal Generation with Linear Models**: Constructing lagged return features and OLS directional prediction.
5. **Vectorized Backtesting**: Incorporating transaction costs, cumulative returns, Sharpe ratio, and maximum drawdown.
6. **The Overfitting Trap**: In-sample vs. out-of-sample degradation and walk-forward validation.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8' if 'seaborn-v0_8' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
print("Session 1 Environment Initialized.")


## 1. Data Acquisition & Inspection

We load end-of-day financial data directly from `https://hilpisch.com/eod_data.csv`, which contains historical series for **SPY** (S&P 500 ETF), **EURUSD**, **BTC-USD**, and major single stocks.


In [ ]:
DATA_URL = "https://hilpisch.com/eod_data.csv"

# Load daily price data
df = pd.read_csv(DATA_URL, parse_dates=['Date']).set_index('Date').sort_index()
print(f"Dataset Shape: {df.shape}")
print(f"Date Range: {df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}")
print(f"Available Assets: {list(df.columns)}")
df.head()


## 2. The Efficient Market Hypothesis: Random Walk vs. Real Markets

Under the Weak-Form EMH, future returns cannot be forecasted from historical price patterns.
Let us simulate a **Geometric Brownian Motion (GBM)** matching empirical mean and volatility, then contrast it with real market dynamics.


In [ ]:
symbol = 'SPY'
prices = df[symbol].dropna()
log_returns = np.log(prices / prices.shift(1)).dropna()

# Parameters matching empirical SPY
mu = log_returns.mean()
sigma = log_returns.std()
n_days = len(log_returns)

# Simulate Geometric Brownian Motion (Null Model)
np.random.seed(42)
sim_returns = np.random.normal(mu, sigma, n_days)
sim_prices = prices.iloc[0] * np.exp(np.cumsum(sim_returns))
sim_series = pd.Series(sim_prices, index=prices.index[1:])

# Plot Real vs Simulated
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

axes[0, 0].plot(prices, color='#2F80ED', label=f'Empirical {symbol}')
axes[0, 0].set_title(f'Real Market Price: {symbol}')
axes[0, 0].legend()

axes[0, 1].plot(sim_series, color='#EB5757', label='Simulated Random Walk (GBM)')
axes[0, 1].set_title('Simulated Null Model (GBM)')
axes[0, 1].legend()

axes[1, 0].hist(log_returns, bins=75, density=True, alpha=0.6, color='#2F80ED', label='Empirical')
x_grid = np.linspace(log_returns.min(), log_returns.max(), 500)
axes[1, 0].plot(x_grid, stats.norm.pdf(x_grid, mu, sigma), 'r--', label='Normal Fit')
axes[1, 0].set_title(f'Empirical Return Distribution (Kurtosis: {stats.kurtosis(log_returns):.2f})')
axes[1, 0].legend()

axes[1, 1].hist(sim_returns, bins=75, density=True, alpha=0.6, color='#EB5757', label='Simulated')
axes[1, 1].plot(x_grid, stats.norm.pdf(x_grid, mu, sigma), 'k--', label='Normal Fit')
axes[1, 1].set_title(f'Simulated Return Distribution (Kurtosis: {stats.kurtosis(sim_returns):.2f})')
axes[1, 1].legend()

plt.tight_layout()
plt.show()


## 3. Statistical Tests for Market Predictability

We test for serial dependence using:
- **Autocorrelation Analysis** across multiple lags.
- **Ljung-Box Test** for joint statistical significance.
- **Rolling 1-Year Autocorrelation** to identify temporal regime changes.


In [ ]:
lags_to_test = 10
autocorrs = [log_returns.autocorr(lag=l) for l in range(1, lags_to_test + 1)]

# Ljung-Box Test
lb_test = acorr_ljungbox(log_returns, lags=lags_to_test, return_df=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].bar(range(1, lags_to_test + 1), autocorrs, color='#2F80ED', alpha=0.8)
axes[0].axhline(0, color='black', linewidth=0.8)
conf_bound = 1.96 / np.sqrt(len(log_returns))
axes[0].axhline(conf_bound, color='red', linestyle='--', label='95% Confidence Limit')
axes[0].axhline(-conf_bound, color='red', linestyle='--')
axes[0].set_title(f'{symbol} Return Autocorrelation by Lag')
axes[0].set_xlabel('Lag (Days)')
axes[0].set_ylabel('Autocorrelation')
axes[0].legend()

# Rolling autocorrelation (252-day window)
rolling_ac1 = log_returns.rolling(252).apply(lambda s: s.autocorr(1))
axes[1].plot(rolling_ac1, color='#27AE60', label='1-Year Rolling Lag-1 Autocorrelation')
axes[1].axhline(0, color='black', linestyle='--', linewidth=0.8)
axes[1].set_title('Regime Variation: Rolling 1-Year Lag-1 Autocorrelation')
axes[1].legend()

plt.tight_layout()
plt.show()

print("Ljung-Box Diagnostic Results:")
print(lb_test)


## 4. Signal Generation with Linear OLS Regression

We construct a lagged return matrix $X_t = [r_{t-1}, r_{t-2}, \dots, r_{t-5}]$ to forecast directional sign $	ext{sign}(\hat{r}_{t})$.


In [ ]:
lags = 5
data = pd.DataFrame(index=prices.index)
data['price'] = prices
data['return'] = log_returns

cols = []
for lag in range(1, lags + 1):
    col = f'lag_{lag}'
    data[col] = data['return'].shift(lag)
    cols.append(col)

data = data.dropna()

# Fit OLS Regression
ols = LinearRegression(fit_intercept=True)
ols.fit(data[cols], data['return'])

data['pred_return'] = ols.predict(data[cols])
data['position_ols'] = np.sign(data['pred_return'])

print(f"OLS Intercept: {ols.intercept_:.6f}")
for col, coef in zip(cols, ols.coef_):
    print(f"OLS Coefficient ({col}): {coef:+.6f}")


## 5. Vectorized Backtesting with Transaction Costs

We simulate trade execution with proportional transaction costs $c = 5	ext{ bps}$ per position turn.


In [ ]:
tc = 0.0005  # 5 bps per trade turnover (0.05%)

# Strategy gross and net returns
data['strategy_gross'] = data['position_ols'] * data['return']
data['trades'] = data['position_ols'].diff().abs().fillna(data['position_ols'].abs())
data['strategy_net'] = data['strategy_gross'] - (data['trades'] * tc)

# Cumulative returns
data['creturns_market'] = np.exp(data['return'].cumsum())
data['creturns_gross'] = np.exp(data['strategy_gross'].cumsum())
data['creturns_net'] = np.exp(data['strategy_net'].cumsum())

# Performance Metrics
ann_factor = 252
years = len(data) / ann_factor
ann_ret_mkt = np.exp(data['return'].sum() / years) - 1.0
ann_ret_net = np.exp(data['strategy_net'].sum() / years) - 1.0
vol_net = data['strategy_net'].std() * np.sqrt(ann_factor)
sharpe_net = ann_ret_net / vol_net

# Drawdown
cum_net = data['creturns_net']
drawdown = (cum_net - cum_net.cummax()) / cum_net.cummax()
max_dd = drawdown.min()

print("--- IN-SAMPLE PERFORMANCE SUMMARY ---")
print(f"Annualized Market Return   : {ann_ret_mkt:6.2%}")
print(f"Annualized Strategy (Net)  : {ann_ret_net:6.2%}")
print(f"Annualized Net Volatility  : {vol_net:6.2%}")
print(f"Sharpe Ratio (Net)         : {sharpe_net:6.2f}")
print(f"Maximum Drawdown           : {max_dd:6.2%}")
print(f"Total Trade Switches       : {int(data['trades'].sum())}")

# Plot In-Sample Backtest
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [3, 1]})
axes[0].plot(data['creturns_market'], label='Buy & Hold (SPY)', color='gray', alpha=0.7)
axes[0].plot(data['creturns_gross'], label='OLS Strategy (Gross)', color='#2F80ED', linestyle='--')
axes[0].plot(data['creturns_net'], label='OLS Strategy (Net of Costs)', color='#27AE60', linewidth=2)
axes[0].set_title('In-Sample Backtest: OLS Lagged Returns Strategy')
axes[0].set_ylabel('Cumulative Growth ($1 Base)')
axes[0].legend()

axes[1].fill_between(drawdown.index, drawdown, 0, color='#EB5757', alpha=0.4, label='Net Drawdown')
axes[1].set_title('Strategy Drawdown Profile')
axes[1].set_ylabel('Drawdown')
axes[1].legend()
plt.tight_layout()
plt.show()


## 6. The Overfitting Trap: Train/Test Split & Out-of-Sample Decay

We split the timeline (60% Train / 40% Test) to examine out-of-sample performance degradation.


In [ ]:
split_idx = int(len(data) * 0.6)
train_df = data.iloc[:split_idx].copy()
test_df = data.iloc[split_idx:].copy()

# Fit strictly on train set
ols_train = LinearRegression().fit(train_df[cols], train_df['return'])

# Out-of-Sample Predictions
test_df['pred_return'] = ols_train.predict(test_df[cols])
test_df['position_oos'] = np.sign(test_df['pred_return'])
test_df['strategy_gross'] = test_df['position_oos'] * test_df['return']
test_df['trades'] = test_df['position_oos'].diff().abs().fillna(test_df['position_oos'].abs())
test_df['strategy_net'] = test_df['strategy_gross'] - (test_df['trades'] * tc)

test_df['creturns_market'] = np.exp(test_df['return'].cumsum())
test_df['creturns_net'] = np.exp(test_df['strategy_net'].cumsum())

oos_years = len(test_df) / ann_factor
oos_ann_net = np.exp(test_df['strategy_net'].sum() / oos_years) - 1.0
oos_vol_net = test_df['strategy_net'].std() * np.sqrt(ann_factor)
oos_sharpe = oos_ann_net / oos_vol_net if oos_vol_net > 0 else 0

print("--- OUT-OF-SAMPLE (OOS) RESULTS ---")
print(f"OOS Annualized Strategy Net: {oos_ann_net:6.2%}")
print(f"OOS Sharpe Ratio           : {oos_sharpe:6.2f}")
print(f"OOS Accuracy (Sign Match)  : {accuracy_score((test_df['return'] > 0), (test_df['position_oos'] > 0)):6.2%}")

plt.figure(figsize=(14, 6))
plt.plot(test_df['creturns_market'], label='Buy & Hold (SPY Test Period)', color='gray')
plt.plot(test_df['creturns_net'], label='OLS Strategy (OOS Net)', color='#2F80ED', linewidth=2)
plt.title('Out-of-Sample (OOS) Performance Evaluation')
plt.ylabel('Cumulative Growth')
plt.legend()
plt.show()


---
### Summary & Transition to Session 2
- Pure linear models capture linear correlation, but real market interactions are non-linear, state-dependent, and multi-factor.
- In **Session 2**, we leverage **PyTorch and Colab GPUs** to train deep neural networks with non-linear feature transformations and confidence thresholds.
